In [1]:
!pip install langchain-experimental langchain-openai pandas requests==2.32.4
!pip install gdown

In [5]:
import pandas as pd
import os
import sys
import gdown
from getpass import getpass
from langchain_openai import ChatOpenAI
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
from google.colab import userdata

In [3]:
files_to_download = {
    "saas_docs.csv":         "https://drive.google.com/file/d/1RElOhN7bYsDAJUNQhYyqM7IzX-Xo6myq/view?usp=sharing",
    "credit_card_terms.csv": "https://drive.google.com/file/d/1_giivc_B0urOKpct0XY2yVZuxW3Eenuf/view?usp=sharing",
    "hospital_policy.csv":   "https://drive.google.com/file/d/1pL7OnDhnmz9pteIpBJ12gu2_ixrc2hPm/view?usp=sharing",
    "ecommerce_faqs.csv":    "https://drive.google.com/file/d/1O4fTjsLFbz55oOiwJUwLwZryO5OSSF6p/view?usp=sharing"
}

In [4]:
print("--- Downloading Files from Google Drive ---")
for filename, url in files_to_download.items():
    if not os.path.exists(filename):
        gdown.download(url, filename, quiet=False, fuzzy=True)
        print(f"Downloaded: {filename}")
    else:
        print(f"Skipped: {filename} (Already exists)")
print("--- Download Complete ---\n")

--- Downloading Files from Google Drive ---


Downloading...
From: https://drive.google.com/uc?id=1RElOhN7bYsDAJUNQhYyqM7IzX-Xo6myq
To: /content/saas_docs.csv
100%|██████████| 2.84k/2.84k [00:00<00:00, 9.72MB/s]


Downloaded: saas_docs.csv


Downloading...
From: https://drive.google.com/uc?id=1_giivc_B0urOKpct0XY2yVZuxW3Eenuf
To: /content/credit_card_terms.csv
100%|██████████| 2.98k/2.98k [00:00<00:00, 12.1MB/s]


Downloaded: credit_card_terms.csv


Downloading...
From: https://drive.google.com/uc?id=1pL7OnDhnmz9pteIpBJ12gu2_ixrc2hPm
To: /content/hospital_policy.csv
100%|██████████| 3.45k/3.45k [00:00<00:00, 16.7MB/s]


Downloaded: hospital_policy.csv


Downloading...
From: https://drive.google.com/uc?id=1O4fTjsLFbz55oOiwJUwLwZryO5OSSF6p
To: /content/ecommerce_faqs.csv
100%|██████████| 3.52k/3.52k [00:00<00:00, 8.98MB/s]

Downloaded: ecommerce_faqs.csv
--- Download Complete ---



In [6]:
print("ENTER YOUR OPENAI API KEY BELOW:")
#api_key = getpass()
api_key=userdata.get('OPENAI_APIKEY')

ENTER YOUR OPENAI API KEY BELOW:


In [7]:
dataframes = []
loaded_names = []

In [8]:
try:
    for filename in files_to_download.keys():
        df = pd.read_csv(filename)
        dataframes.append(df)
        loaded_names.append(filename)
        print(f"SUCCESS: Loaded '{filename}' ({len(df)} rows)")

except Exception as e:
    print(f"\nERROR loading files: {e}")
    sys.exit()

SUCCESS: Loaded 'saas_docs.csv' (15 rows)
SUCCESS: Loaded 'credit_card_terms.csv' (15 rows)
SUCCESS: Loaded 'hospital_policy.csv' (15 rows)
SUCCESS: Loaded 'ecommerce_faqs.csv' (15 rows)


In [9]:
custom_prefix = """
You are a master data analyst working with 4 specific DataFrames.
Your goal is to query the correct dataframe to answer user questions.

Here is your data map:
- df1: SaaS Documentation (Technical docs, API usage)
- df2: Credit Card Terms (Financial terms, interest rates, fees)
- df3: Hospital Policy (Visiting rules, admin info)
- df4: Ecommerce FAQs (Shipping, returns, product info)

RULES:
1. First, reason about which DataFrame is most relevant to the question.
2. Use Python string methods like `.str.contains('keyword', case=False)` to find relevant rows.
3. DO NOT look for exact matches; always use flexible string matching.
4. If you find the answer, print it clearly.
"""

In [10]:
system_prompt = """
You are a smart data assistant capable of reading multiple CSV files.
- You have access to 4 different datasets: SaaS Docs, Credit Card Terms, Hospital Policy, and Ecommerce FAQs.
- When asked a question, determine which DataFrame is most relevant.
- Do NOT answer from general knowledge.
- Answer in plain English.
"""

In [11]:
try:
    llm = ChatOpenAI(
        temperature=0.0,
        model="gpt-4o-mini",
        openai_api_key=api_key
    )

    # --- KEY CHANGE: We pass the LIST 'dataframes' instead of a single 'df' ---
    agent = create_pandas_dataframe_agent(
       llm,
        dataframes,
        verbose=True,
        agent_type="openai-tools", # 'openai-tools' is often more stable than 'functions'
        prefix=custom_prefix,      # <--- Injecting the map here
        include_df_in_prompt=True, # Ensures the agent sees column names
        handle_parsing_errors=True, # <--- Allows agent to self-correct coding errors
        allow_dangerous_code=True
    )

    print("\nAI Agent is ready! You can ask questions across ALL files.")
    print("Example: 'What is the visiting hour in the hospital?' or 'What is the API limit?'")

except Exception as e:
    print(f"Error initializing agent: {e}")
    sys.exit()


AI Agent is ready! You can ask questions across ALL files.
Example: 'What is the visiting hour in the hospital?' or 'What is the API limit?'


/usr/local/lib/python3.12/dist-packages/langchain_experimental/agents/agent_toolkits/pandas/base.py:283: UserWarning: Received additional kwargs {'handle_parsing_errors': True} which are no longer supported.
  warnings.warn(


In [13]:
while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit", "q"]:
        print("Goodbye!")
        break

    if not user_input:
        continue

    final_query = system_prompt + "\n\nQuestion: " + user_input
    print("AI is thinking...")

    try:
        response = agent.invoke(final_query)['output']
        print(f"AI: {response}\n" + "-"*30)
    except Exception as e:
        print(f"An error occurred: {e}")

You: What is the API limit
AI is thinking...


> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "df1[df1['Description'].str.contains('API limit', case=False)]"}`


Empty DataFrame
Columns: [Doc ID, Feature, Plan Level, Description, Technical Limit, Related API]
Index: []
Invoking: `python_repl_ast` with `{'query': "df1[df1['Description'].str.contains('rate limit', case=False)]"}`


Empty DataFrame
Columns: [Doc ID, Feature, Plan Level, Description, Technical Limit, Related API]
Index: []
Invoking: `python_repl_ast` with `{'query': "df1[df1['Description'].str.contains('limit', case=False)]"}`


  Doc ID         Feature  Plan Level  \
0  S-501  API Rate Limit        Free   
6  S-507    File Storage        Free   
7  S-508    File Storage  Enterprise   

                                         Description Technical Limit  \
0  Users on the Free tier are limited to 1,000 AP...    1000 req/day   
6  Free accounts are limited to 100MB of file sto...        

KeyboardInterrupt: Interrupted by user